In [1]:
# librería de Python que permite conectarse a PostgreSQL:
%pip install psycopg[binary]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 16.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [psycopg]m1/2 [psycopg]
Note: you may need to restart the kernel to use updated packages.


In [28]:
# Creamos la conexión

import psycopg

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="superdata",
    user="postgres",
    password="postgres"
) # este objeto es la conexión de postgresql con python

In [29]:
# Creamos la conexión:

cursor = conn.cursor() # el cursor manda la consulta sql a postgree

# Ejecutamos la consulta SQL
cursor.execute("""
    DROP TABLE clientes;
""") # ejecutamos la consulta

conn.commit() 

In [30]:
# Creamos las tablas:

import polars as pl

clientes = pl.read_csv("../data/clientes.csv")

clientes.head()

id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,str,str,str,i64,str,str,i64,str,bool,str
1,"""Álvaro""","""Castro Navarro""","""Mujer""",24,"""Valencia""","""Valencia""",35314,"""2023-04-25""",false,"""Normal"""
2,"""Miguel""","""Martín Sánchez""","""Mujer""",40,"""Santander""","""Cantabria""",49225,"""2022-01-22""",false,"""Ocasional"""
3,"""Elena""","""Romero Ortega""","""Mujer""",61,"""Alicante""","""Alicante""",23449,"""2023-11-14""",true,"""Normal"""
4,"""María""","""Romero Pérez""","""Hombre""",49,"""Salamanca""","""Salamanca""",18478,"""2024-06-24""",true,"""Normal"""
5,"""Alejandro""","""Jiménez Castro""","""Mujer""",28,"""Logroño""","""La Rioja""",36179,"""2022-11-16""",true,"""Frecuente"""


In [31]:
# Convertir tipos de datos

# Polars y PostgreSQL utilizan sistemas de tipos diferentes.


def polars_to_postgres_type(dtype):
    if dtype in (pl.Int8, pl.Int16, pl.Int32, pl.Int64):
        return "INTEGER"
    elif dtype in (pl.Float32, pl.Float64):
        return "DOUBLE PRECISION"
    elif dtype == pl.Boolean:
        return "BOOLEAN"
    elif dtype == pl.Date:
        return "DATE"
    elif dtype == pl.Datetime:
        return "TIMESTAMP"
    else:
        return "TEXT"

In [32]:
clientes.schema

Schema([('id_cliente', Int64),
        ('nombre', String),
        ('apellidos', String),
        ('sexo', String),
        ('edad', Int64),
        ('ciudad', String),
        ('provincia', String),
        ('codigo_postal', Int64),
        ('fecha_alta', String),
        ('socio', Boolean),
        ('tipo_cliente', String)])

In [33]:
# Construimos la consulta a partir de el csv clientes para crear su tabla

columnas_clientes = []

for nombre, tipo in clientes.schema.items():
    tipo_postgres = polars_to_postgres_type(tipo)

    if nombre == "id_cliente":
        columnas_clientes.append(
            f'"{nombre}" {tipo_postgres} PRIMARY KEY'
        )
    else:
        columnas_clientes.append(
            f'"{nombre}" {tipo_postgres}'
        )

sql_clientes = f"""
CREATE TABLE IF NOT EXISTS clientes (
    {", ".join(columnas_clientes)}
);
"""

print(sql_clientes)


CREATE TABLE IF NOT EXISTS clientes (
    "id_cliente" INTEGER PRIMARY KEY, "nombre" TEXT, "apellidos" TEXT, "sexo" TEXT, "edad" INTEGER, "ciudad" TEXT, "provincia" TEXT, "codigo_postal" INTEGER, "fecha_alta" TEXT, "socio" BOOLEAN, "tipo_cliente" TEXT
);



In [34]:
cursor.execute(sql_clientes)

conn.commit()

In [35]:
# Hay que introducir los clientes en la tabla que hemos creado

columnas = clientes.columns

placeholders = ", ".join(["%s"] * len(columnas)) # Los `%s` representan los valores que proporcionaremos posteriormente.

sql_insert_clientes = f"""
INSERT INTO clientes ({", ".join(columnas)})
VALUES ({placeholders})
"""

print(sql_insert_clientes)


INSERT INTO clientes (id_cliente, nombre, apellidos, sexo, edad, ciudad, provincia, codigo_postal, fecha_alta, socio, tipo_cliente)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)



In [36]:
# Ahora sí le pasamos las filas:

filas_clientes = clientes.rows()

cursor.executemany(
    sql_insert_clientes,
    filas_clientes
)

conn.commit()

print(f"Se han insertado {len(filas_clientes)} clientes")

Se han insertado 500 clientes


In [11]:
# Compronamos que todo ha ido bien:

cursor.execute("""
    SELECT *
    FROM clientes
    LIMIT 10
""")

resultados = cursor.fetchall() # traemos todos los resultados

resultados

[(1,
  'Álvaro',
  'Castro Navarro',
  'Mujer',
  24,
  'Valencia',
  'Valencia',
  35314,
  '2023-04-25',
  False,
  'Normal'),
 (2,
  'Miguel',
  'Martín Sánchez',
  'Mujer',
  40,
  'Santander',
  'Cantabria',
  49225,
  '2022-01-22',
  False,
  'Ocasional'),
 (3,
  'Elena',
  'Romero Ortega',
  'Mujer',
  61,
  'Alicante',
  'Alicante',
  23449,
  '2023-11-14',
  True,
  'Normal'),
 (4,
  'María',
  'Romero Pérez',
  'Hombre',
  49,
  'Salamanca',
  'Salamanca',
  18478,
  '2024-06-24',
  True,
  'Normal'),
 (5,
  'Alejandro',
  'Jiménez Castro',
  'Mujer',
  28,
  'Logroño',
  'La Rioja',
  36179,
  '2022-11-16',
  True,
  'Frecuente'),
 (6,
  'Daniel',
  'Jiménez Sánchez',
  'Mujer',
  39,
  'Málaga',
  'Málaga',
  52769,
  '2022-05-06',
  True,
  'Frecuente'),
 (7,
  'Claudia',
  'Ruiz López',
  'Hombre',
  39,
  'Zaragoza',
  'Zaragoza',
  47881,
  '2022-10-04',
  False,
  'Frecuente'),
 (8,
  'Javier',
  'Ortega García',
  'Mujer',
  60,
  'Alicante',
  'Alicante',
  26338,
  

In [12]:
# Podemos volver a convertir los daros a polars:

clientes_db = pl.DataFrame(
    resultados,
    schema=clientes.columns,
    orient="row"
)

clientes_db

id_cliente,nombre,apellidos,sexo,edad,ciudad,provincia,codigo_postal,fecha_alta,socio,tipo_cliente
i64,str,str,str,i64,str,str,i64,str,bool,str
1,"""Álvaro""","""Castro Navarro""","""Mujer""",24,"""Valencia""","""Valencia""",35314,"""2023-04-25""",false,"""Normal"""
2,"""Miguel""","""Martín Sánchez""","""Mujer""",40,"""Santander""","""Cantabria""",49225,"""2022-01-22""",false,"""Ocasional"""
3,"""Elena""","""Romero Ortega""","""Mujer""",61,"""Alicante""","""Alicante""",23449,"""2023-11-14""",true,"""Normal"""
4,"""María""","""Romero Pérez""","""Hombre""",49,"""Salamanca""","""Salamanca""",18478,"""2024-06-24""",true,"""Normal"""
5,"""Alejandro""","""Jiménez Castro""","""Mujer""",28,"""Logroño""","""La Rioja""",36179,"""2022-11-16""",true,"""Frecuente"""
6,"""Daniel""","""Jiménez Sánchez""","""Mujer""",39,"""Málaga""","""Málaga""",52769,"""2022-05-06""",true,"""Frecuente"""
7,"""Claudia""","""Ruiz López""","""Hombre""",39,"""Zaragoza""","""Zaragoza""",47881,"""2022-10-04""",false,"""Frecuente"""
8,"""Javier""","""Ortega García""","""Mujer""",60,"""Alicante""","""Alicante""",26338,"""2022-07-13""",true,"""Ocasional"""
9,"""Claudia""","""Jiménez Pérez""","""Hombre""",42,"""Gijón""","""Asturias""",27899,"""2023-11-27""",true,"""Normal"""


In [13]:
# Ejemplo 1: Contar el num de clientes

cursor.execute("""
    SELECT COUNT(*)
    FROM clientes
""")

numero_clientes = cursor.fetchone()[0] # es para traer un resultado

print(f"Clientes almacenados: {numero_clientes}")

Clientes almacenados: 500


In [14]:
# Para crear nuevas tablas vamos a crear una función:

def crear_tabla_desde_polars(
    cursor,
    conn,
    df,
    nombre_tabla,
    clave_primaria=None
):
    columnas = []

    for nombre, tipo in df.schema.items():

        tipo_postgres = polars_to_postgres_type(tipo)

        columna = f'"{nombre}" {tipo_postgres}'

        if nombre == clave_primaria:
            columna += " PRIMARY KEY"

        columnas.append(columna)

    sql = f"""
    CREATE TABLE IF NOT EXISTS "{nombre_tabla}" (
        {", ".join(columnas)}
    );
    """

    cursor.execute(sql)
    conn.commit()

    print(f"Tabla '{nombre_tabla}' creada.")

In [15]:
# Vamos a usarla para crear las tablas de productos y ventas:

productos = pl.read_csv("../data/productos.csv")
ventas = pl.read_csv("../data/ventas.csv")

print("Productos:")
print(productos.shape)

print("\nVentas:")
print(ventas.shape)

Productos:
(150, 6)

Ventas:
(10000, 7)


In [16]:
crear_tabla_desde_polars(
    cursor,
    conn,
    productos,
    "productos",
    "id_producto"
)

crear_tabla_desde_polars(
    cursor,
    conn,
    ventas,
    "ventas",
    "id_venta"
)

Tabla 'productos' creada.
Tabla 'ventas' creada.


In [17]:
# Hacemos tb una función para insertar los datos a las tablas

def insertar_dataframe(cursor, conn, df, nombre_tabla):

    columnas = df.columns

    placeholders = ", ".join(
        ["%s"] * len(columnas)
    )

    sql = f"""
    INSERT INTO "{nombre_tabla}"
    ({", ".join(columnas)})
    VALUES ({placeholders})
    """

    cursor.executemany(
        sql,
        df.rows()
    )

    conn.commit()

    print(
        f"Se han insertado "
        f"{df.height} registros en '{nombre_tabla}'."
    )

In [18]:
# Insertamos

insertar_dataframe(
    cursor,
    conn,
    productos,
    "productos"
)

insertar_dataframe(
    cursor,
    conn,
    ventas,
    "ventas"
)

error ignored terminating <psycopg.Pipeline [INERROR, pipeline=ON] (host=localhost user=postgres database=superdata) at 0x7fdf98ceb750>: pipeline aborted


UniqueViolation: duplicate key value violates unique constraint "productos_pkey"
DETAIL:  Key (id_producto)=(1) already exists.

In [31]:
# Ejemplo 2: comprobación de ventas

cursor.execute("""
    SELECT *
    FROM ventas
    LIMIT 1
""")

venta = cursor.fetchone()

venta

(1, '2025-04-07 09:00:00', 276, 1, 3, 'Tarjeta', 10)

In [32]:
# Ejemplo 3: consulta de productos

cursor.execute("""
    SELECT
        id_producto,
        nombre,
        categoria,
        precio,
        coste
    FROM productos
    LIMIT 10
""")

productos_db = cursor.fetchall()

productos_db

[(1, 'Barra', 'Panadería', 8.49, 4.38),
 (2, 'Snacks perro', 'Mascotas', 9.92, 6.73),
 (3, 'Detergente', 'Limpieza', 5.45, 2.48),
 (4, 'Calamares', 'Pescados', 14.94, 9.11),
 (5, 'Turrón', 'Dulces', 4.37, 2.18),
 (6, 'Toallitas', 'Bebé', 6.99, 5.09),
 (7, 'Papel higiénico', 'Higiene', 4.47, 2.17),
 (8, 'Brócoli', 'Verduras', 0.92, 0.61),
 (9, 'Patata', 'Verduras', 5.73, 3.27),
 (10, 'Leche infantil', 'Bebé', 6.84, 3.98)]

In [33]:
# Ejemplo 4: ¿Cuáles son los productos cuyo precio es superior a 20 €?

cursor.execute("""
    SELECT
        id_producto,
        nombre,
        categoria,
        precio
    FROM productos
    WHERE precio > 20
""")

productos_caros = cursor.fetchall()

productos_caros

[]

In [34]:
# Ejemplo 5: Ventas por método de pago:

cursor.execute("""
    SELECT
        metodo_pago,
        COUNT(*) AS numero_ventas
    FROM ventas
    GROUP BY metodo_pago
""")

ventas_pago = cursor.fetchall()

ventas_pago

[('Bizum', 2005),
 ('Cheque regalo', 474),
 ('Efectivo', 1495),
 ('Tarjeta', 6026)]

In [38]:
# Ejercicio: Plantea 5 consultas y ejecútalas con SQL.

# Notebook 11 con postgres_utils.py

In [35]:
from postgres_utils import (
    conectar,
    cargar_datos,
    crear_tabla_desde_polars,
    insertar_dataframe
)

In [36]:
conn = conectar()

In [37]:
clientes, productos, ventas = cargar_datos()

In [38]:
crear_tabla_desde_polars(
    conn,
    clientes,
    "clientes",
    "id_cliente"
)

crear_tabla_desde_polars(
    conn,
    productos,
    "productos",
    "id_producto"
)

crear_tabla_desde_polars(
    conn,
    ventas,
    "ventas",
    "id_venta"
)

In [39]:
insertar_dataframe(conn, clientes, "clientes")
insertar_dataframe(conn, productos, "productos")
insertar_dataframe(conn, ventas, "ventas")

error ignored terminating <psycopg.Pipeline [INERROR, pipeline=ON] (host=localhost user=postgres database=superdata) at 0x7f82e407a9f0>: pipeline aborted


UniqueViolation: duplicate key value violates unique constraint "clientes_pkey"
DETAIL:  Key (id_cliente)=(1) already exists.